# VM-local CSV MLP Classification and Unlearning Pipeline

# The two processed datasets are read directly from local CSV files. Each row is
# `[label_id, relative_time, direction, packet_size]` repeated 10,000 times.
# Raw data are never modified; all cache and experiment outputs are local.




In [1]:
# OVERVIEW: Configure local CSV datasets, artifact paths, GPU, and experiment controls.
from pathlib import Path
import os

RUN_ID = 'aol_known_vs_273_unknown_csv_seed42'

# TODO(user): these are the complete local VM paths supplied for the two sources.
AOL_DATASET_DIR = Path('/home/ubuntu/Documents/KF/data_processed/AOL_csv/icloud')
UNKNOWN_273_DATASET_DIR = Path('/home/ubuntu/Documents/KF/data_processed/icloud_100')

PRETRAIN_AOL_PATH = Path('MLP-Classfication/vm_code/weight_trained/pretrain_AOL.pth')
if not PRETRAIN_AOL_PATH.exists():
    PRETRAIN_AOL_PATH = Path('weight_trained/pretrain_AOL.pth')

OUTPUT_DIR = Path.cwd() / 'artifacts' / 'vm-training' / 'experiments' / RUN_ID
FEATURE_CACHE_DIR = OUTPUT_DIR / 'feature_cache'
INDEX_PATH = OUTPUT_DIR / 'csv_index.json'
BASE_MODEL_DIR = OUTPUT_DIR / 'base_model'
UNLEARNING_ROOT = OUTPUT_DIR / 'unlearning'

MAX_PACKETS = 10_000
RAW_FIELDS_PER_PACKET = 4       # [label_id, relative_time, direction, packet_size]
PACKET_FEATURES = 3             # [relative_time, direction, packet_size]
ROW_WIDTH = MAX_PACKETS * RAW_FIELDS_PER_PACKET
FEATURE_TRANSFORM = 'legacy_raw_v1'

SEED = 42
VALID_RATIO = 0.15
BATCH_SIZE = 16
NUM_WORKERS = 4
EPOCHS = 16
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
UNKNOWN_THRESHOLD = 0.50
THRESHOLD_GRID = (0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70)
FREEZE_ENCODER_FOR_BASE_TRAINING = True
CACHE_FEATURES = True
DEVICE_NAME = 'cuda:0'
REQUIRE_CUDA = True

# Phase 3. Run base training first with False, then set True to produce three baselines.
RUN_UNLEARNING = True
FORGET_LABEL = None
UNLEARNING_EPOCHS = 3
UNLEARNING_LR = 1e-4
RETAIN_LOSS_WEIGHT = 1.0
UNLEARNING_SCOPES = ('head_only', 'last_encoder_block', 'full_encoder_and_head')

print('AOL known:', AOL_DATASET_DIR)
print('273 unknown:', UNKNOWN_273_DATASET_DIR)
print('Output:', OUTPUT_DIR)




AOL known: /home/ubuntu/Documents/KF/data_processed/AOL_csv/icloud
273 unknown: /home/ubuntu/Documents/KF/data_processed/icloud_100
Output: /home/ubuntu/Documents/huyenchi/BKCS-unlearning/MLP-Classfication/vm_code/artifacts/vm-training/experiments/aol_known_vs_273_unknown_csv_seed42


In [2]:
# OVERVIEW: Import libraries and validate the VM-local CSV layout and GPU.
from __future__ import annotations

import csv
import copy
import hashlib
import json
import random
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from typing import Any, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader, Dataset

for root in (AOL_DATASET_DIR, UNKNOWN_273_DATASET_DIR):
    for name in ('train_data.csv', 'test_data.csv', 'label_mapping.csv'):
        if not (root / name).is_file():
            raise FileNotFoundError(f'Missing {name}: {root / name}')
if not PRETRAIN_AOL_PATH.is_file():
    raise FileNotFoundError(f'Missing pre-trained encoder: {PRETRAIN_AOL_PATH}')

DEVICE = torch.device(DEVICE_NAME if DEVICE_NAME else ('cuda' if torch.cuda.is_available() else 'cpu'))
if REQUIRE_CUDA and DEVICE.type != 'cuda':
    raise RuntimeError('CUDA is required. Use a GPU VM or set REQUIRE_CUDA=False only for debugging.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE, '| CUDA:', torch.cuda.is_available())




Device: cuda:0 | CUDA: True


In [3]:
# OVERVIEW: Build/reuse a byte-offset index and lazily decode one CSV sample at a time.
@dataclass(frozen=True)
class CsvRecord:
    csv_path: str
    byte_offset: int
    line_number: int
    source: str
    original_label: str
    original_label_id: int
    binary_label: int
    split_file: str

def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')

def read_label_mapping(path: Path) -> dict[int, str]:
    with path.open(newline='', encoding='utf-8') as handle:
        rows = csv.DictReader(handle)
        return {int(row['label_id']): (row['label_name'] or f'label_{row["label_id"]}') for row in rows}

def file_signature(path: Path) -> dict[str, Any]:
    stat = path.stat()
    return {'path': str(path.resolve()), 'bytes': stat.st_size, 'mtime_ns': stat.st_mtime_ns}

def index_one_csv(csv_path: Path, mapping: dict[int, str], source: str, binary_label: int, split_file: str) -> list[CsvRecord]:
    records: list[CsvRecord] = []
    with csv_path.open('rb') as handle:
        line_number = 0
        while True:
            byte_offset = handle.tell()
            raw = handle.readline()
            if not raw: break
            line_number += 1
            first = raw.split(b',', 1)[0]
            try:
                label_id = int(float(first))
            except ValueError as error:
                raise ValueError(f'Invalid label at {csv_path}:{line_number}') from error
            records.append(CsvRecord(str(csv_path), byte_offset, line_number, source, mapping.get(label_id, f'label_{label_id}'), label_id, binary_label, split_file))
    if not records: raise ValueError(f'CSV has no samples: {csv_path}')
    return records

def load_or_build_index() -> tuple[list[CsvRecord], list[CsvRecord]]:
    paths = [root / name for root in (AOL_DATASET_DIR, UNKNOWN_273_DATASET_DIR) for name in ('train_data.csv', 'test_data.csv', 'label_mapping.csv')]
    signatures = [file_signature(path) for path in paths]
    if INDEX_PATH.is_file():
        cached = json.loads(INDEX_PATH.read_text(encoding='utf-8'))
        if cached.get('signatures') == signatures:
            return [CsvRecord(**row) for row in cached['train']], [CsvRecord(**row) for row in cached['test']]
    aol_mapping = read_label_mapping(AOL_DATASET_DIR / 'label_mapping.csv')
    unknown_mapping = read_label_mapping(UNKNOWN_273_DATASET_DIR / 'label_mapping.csv')
    aol_train = index_one_csv(AOL_DATASET_DIR / 'train_data.csv', aol_mapping, 'AOL', 0, 'train_data.csv')
    unknown_train = index_one_csv(UNKNOWN_273_DATASET_DIR / 'train_data.csv', unknown_mapping, '273', 1, 'train_data.csv')
    aol_test = index_one_csv(AOL_DATASET_DIR / 'test_data.csv', aol_mapping, 'AOL', 0, 'test_data.csv')
    unknown_test = index_one_csv(UNKNOWN_273_DATASET_DIR / 'test_data.csv', unknown_mapping, '273', 1, 'test_data.csv')
    train, test = aol_train + unknown_train, aol_test + unknown_test
    write_json(INDEX_PATH, {'signatures': signatures, 'train': [asdict(row) for row in train], 'test': [asdict(row) for row in test]})
    return train, test

def cache_path(record: CsvRecord) -> Path:
    key = hashlib.sha1(f'{record.csv_path}:{record.byte_offset}:{ROW_WIDTH}:legacy_raw_v1'.encode()).hexdigest()
    return FEATURE_CACHE_DIR / f'{key}.pt'

def decode_csv_record(record: CsvRecord) -> tuple[Tensor, Tensor]:
    with open(record.csv_path, 'rb') as handle:
        handle.seek(record.byte_offset)
        raw = handle.readline()
    values = np.fromstring(raw.decode('ascii').strip(), sep=',', dtype=np.float32)
    if values.size != ROW_WIDTH:
        raise ValueError(f'Expected {ROW_WIDTH} values at {record.csv_path}:{record.line_number}, got {values.size}')
    packets = values.reshape(MAX_PACKETS, RAW_FIELDS_PER_PACKET)
    labels = packets[:, 0]
    observed = labels[labels != 0]
    if observed.size and not np.allclose(observed, float(record.original_label_id)):
        raise ValueError(f'Inconsistent repeated label at {record.csv_path}:{record.line_number}')
    features = torch.from_numpy(packets[:, 1:].copy())
    mask = features[:, 2].ne(0)  # packet_size=0 only for zero padding
    return features, mask

def load_features(record: CsvRecord) -> tuple[Tensor, Tensor]:
    path = cache_path(record)
    if CACHE_FEATURES and path.is_file():
        payload = torch.load(path, map_location='cpu', weights_only=True)
        return payload['features'], payload['mask']
    features, mask = decode_csv_record(record)
    if CACHE_FEATURES:
        temporary = path.with_suffix('.tmp')
        torch.save({'features': features, 'mask': mask}, temporary)
        temporary.replace(path)
    return features, mask

class CsvFlowDataset(Dataset):
    def __init__(self, records: Sequence[CsvRecord]):
        self.records = list(records)
        self.labels = torch.tensor([row.binary_label for row in records], dtype=torch.long)
    def __len__(self) -> int: return len(self.records)
    def __getitem__(self, index: int):
        record = self.records[index]
        features, mask = load_features(record)
        return features, mask, torch.tensor(record.binary_label), record.original_label

def make_loader(records: Sequence[CsvRecord], shuffle: bool) -> DataLoader:
    return DataLoader(CsvFlowDataset(records), batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda', persistent_workers=NUM_WORKERS > 0)

def move(batch):
    return tuple(item.to(DEVICE, non_blocking=True) for item in batch[:3])




In [4]:
# OVERVIEW: Define the checkpoint-compatible legacy RawPacketEncoder and a new binary MLP head.
class FlowEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        def block(cin, cout, suffix):
            setattr(self, f'conv{suffix}', nn.Conv1d(cin, cout, 8))
            setattr(self, f'conv{suffix}_{suffix}', nn.Conv1d(cout, cout, 8))
            setattr(self, f'batch_norm{suffix}', nn.BatchNorm1d(cout))
            setattr(self, f'max_pool_{suffix}', nn.MaxPool1d(8, stride=4))
            setattr(self, f'dropout{suffix}', nn.Dropout(.1))
        block(3, 32, 1); block(32, 64, 2); block(64, 128, 3); block(128, 256, 4)
        with torch.no_grad(): flat = self._convs(torch.zeros(1, 3, MAX_PACKETS)).flatten(1).shape[1]
        self.fc = nn.Linear(flat, 256)
    def _one(self, x, i, first=False):
        x = F.pad(x, (3,4)); x = F.elu(getattr(self, f'conv{i}')(x)) if first else F.relu(getattr(self, f'conv{i}')(x))
        x = F.pad(x, (3,4)); x = F.elu(getattr(self, f'batch_norm{i}')(getattr(self, f'conv{i}_{i}')(x))) if first else F.relu(getattr(self, f'batch_norm{i}')(getattr(self, f'conv{i}_{i}')(x)))
        return getattr(self, f'dropout{i}')(getattr(self, f'max_pool_{i}')(F.pad(x, (3,4))))
    def _convs(self, x):
        for i in range(1,5): x = self._one(x, i, first=i == 1)
        return x
    def forward(self, features, mask=None):
        if mask is not None: features = features * mask.unsqueeze(-1)
        return self.fc(self._convs(features.transpose(1,2)).flatten(1))

class FlowModel(nn.Module):
    def __init__(self):
        super().__init__(); self.encoder = FlowEncoder()
        layers=[]; width=256
        for next_width in (512,256,128,64,32): layers += [nn.Linear(width,next_width), nn.LayerNorm(next_width), nn.GELU(), nn.Dropout(.2)]; width=next_width
        self.classifier = nn.Sequential(*layers, nn.Linear(width, 2))
    def forward(self, features, mask=None):
        embedding = self.encoder(features, mask); return self.classifier(embedding), embedding

def load_pretrained_encoder(model: FlowModel) -> dict[str, Any]:
    checkpoint = torch.load(PRETRAIN_AOL_PATH, map_location='cpu', weights_only=True)
    config = checkpoint.get('model_config', {})
    required = {'max_packets': 10000, 'hidden_size': 256, 'embedding_size': 128}
    if any(config.get(k) != v for k,v in required.items()): raise ValueError(f'Incompatible checkpoint config: {config}')
    state = {k.removeprefix('encoder.'): v for k,v in checkpoint['model_state_dict'].items() if k.startswith('encoder.')}
    model.encoder.load_state_dict(state, strict=True)
    return {'path': str(PRETRAIN_AOL_PATH), 'epoch': checkpoint.get('epoch'), 'config': config}




In [5]:
# OVERVIEW: Create deterministic train/validation/test records from CSV source files.
seed_everything(SEED)
source_train_records, test_records = load_or_build_index()
groups: dict[tuple[str,str], list[CsvRecord]] = defaultdict(list)
for record in source_train_records: groups[(record.source, record.original_label)].append(record)
train_records=[]; val_records=[]
rng=random.Random(SEED)
for key, items in sorted(groups.items()):
    items=list(items); rng.shuffle(items)
    n_val=max(1, round(len(items)*VALID_RATIO)) if len(items) >= 3 else 0
    val_records.extend(items[:n_val]); train_records.extend(items[n_val:])
if not all((train_records, val_records, test_records)): raise ValueError('Empty train/validation/test split.')
KNOWN_LABELS={row.original_label for row in source_train_records if row.source == 'AOL'}
UNKNOWN_LABELS={row.original_label for row in source_train_records if row.source == '273'}
write_json(OUTPUT_DIR/'split_manifest.json', {'train':[asdict(x) for x in train_records], 'validation':[asdict(x) for x in val_records], 'test':[asdict(x) for x in test_records]})
print('Records:', len(train_records), len(val_records), len(test_records))
print('Binary train:', Counter(x.binary_label for x in train_records))
train_loader, val_loader, test_loader = make_loader(train_records, True), make_loader(val_records, False), make_loader(test_records, False)




Records: 12659 2233 14895
Binary train: Counter({1: 8499, 0: 4160})


In [6]:
# OVERVIEW: Train MLP on AOL-known + 273-unknown, select validation threshold, and save a reusable base model.
def evaluate(model, loader, threshold=UNKNOWN_THRESHOLD):
    model.eval(); total=correct=known_total=known_ok=unknown_total=unknown_ok=0; loss_sum=0.; criterion=nn.CrossEntropyLoss()
    with torch.no_grad():
        for batch in loader:
            features, mask, labels=move(batch); logits,_=model(features,mask); loss_sum += criterion(logits,labels).item()*labels.numel()
            pred=(torch.softmax(logits,1)[:,1] >= threshold).long(); total += labels.numel(); correct += (pred == labels).sum().item()
            known=labels==0; unknown=labels==1; known_total+=known.sum().item(); unknown_total+=unknown.sum().item(); known_ok+=((pred==labels)&known).sum().item(); unknown_ok+=((pred==labels)&unknown).sum().item()
    known_recall=known_ok/max(known_total,1); unknown_recall=unknown_ok/max(unknown_total,1)
    return {'loss':loss_sum/max(total,1),'accuracy':correct/max(total,1),'known_recall':known_recall,'unknown_recall':unknown_recall,'balanced_accuracy':.5*(known_recall+unknown_recall),'total':total}

model=FlowModel().to(DEVICE)
existing_base_path=BASE_MODEL_DIR/'best_model.pt'
if RUN_UNLEARNING and existing_base_path.is_file():
    existing_base=torch.load(existing_base_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(existing_base['model_state_dict'], strict=True)
    pretrain_info=existing_base.get('pretrain', {'path': str(PRETRAIN_AOL_PATH)})
    EPOCHS=0  # Dùng base model đã lưu; không train lại trước khi chạy unlearning.
    print(f'Reuse base model for unlearning: {existing_base_path}')
else:
    pretrain_info=load_pretrained_encoder(model)
if FREEZE_ENCODER_FOR_BASE_TRAINING:
    for parameter in model.encoder.parameters(): parameter.requires_grad=False
optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
counts=torch.bincount(torch.tensor([x.binary_label for x in train_records]), minlength=2).float(); weights=(counts.sum()/(2*counts)).to(DEVICE)
criterion=nn.CrossEntropyLoss(weight=weights)
best_state=copy.deepcopy(model.state_dict()); best_score=-1.
for epoch in range(1,EPOCHS+1):
    model.train()
    if FREEZE_ENCODER_FOR_BASE_TRAINING: model.encoder.eval()
    for batch in train_loader:
        features,mask,labels=move(batch); optimizer.zero_grad(set_to_none=True); logits,_=model(features,mask); loss=criterion(logits,labels); loss.backward(); optimizer.step()
    metrics=evaluate(model,val_loader); print(f'Epoch {epoch}/{EPOCHS}: val_bal_acc={metrics["balanced_accuracy"]:.4f}')
    if metrics['balanced_accuracy'] > best_score: best_score=metrics['balanced_accuracy']; best_state=copy.deepcopy(model.state_dict())
model.load_state_dict(best_state)
threshold_metrics=[(threshold,evaluate(model,val_loader,threshold)) for threshold in THRESHOLD_GRID]
BEST_UNKNOWN_THRESHOLD=max(threshold_metrics,key=lambda row:row[1]['balanced_accuracy'])[0]
print(f'BEST_UNKNOWN_THRESHOLD={BEST_UNKNOWN_THRESHOLD:.2f}')
test_metrics=evaluate(model,test_loader,BEST_UNKNOWN_THRESHOLD)
payload={'model_state_dict':model.state_dict(),'pretrain':pretrain_info,'best_unknown_threshold':BEST_UNKNOWN_THRESHOLD,'test_metrics':test_metrics,'feature_contract':'[10000,3] legacy raw CSV','known_labels':sorted(KNOWN_LABELS),'unknown_labels':sorted(UNKNOWN_LABELS)}
BASE_MODEL_DIR.mkdir(parents=True,exist_ok=True); torch.save(payload,BASE_MODEL_DIR/'best_model.pt')
write_json(OUTPUT_DIR/'run_summary.json', {'pretrain':pretrain_info,'threshold':BEST_UNKNOWN_THRESHOLD,'test_metrics':test_metrics,'counts':{'train':len(train_records),'validation':len(val_records),'test':len(test_records)}})
(BASE_MODEL_DIR/'README.md').write_text('# Base model\n\nAOL is known (0); 273 is unknown (1). Input comes from legacy CSV rows reshaped `[10000,4]`, dropping the repeated label column to form `[10000,3]`. `best_model.pt` includes the MLP, encoder, threshold, and source label maps.\n',encoding='utf-8')
print('Base test:',test_metrics)




Reuse base model for unlearning: /home/ubuntu/Documents/huyenchi/BKCS-unlearning/MLP-Classfication/vm_code/artifacts/vm-training/experiments/aol_known_vs_273_unknown_csv_seed42/base_model/best_model.pt
BEST_UNKNOWN_THRESHOLD=0.50
Base test: {'loss': 0.19057697411097843, 'accuracy': 0.9520644511581068, 'known_recall': 0.8888661899897855, 'unknown_recall': 0.983, 'balanced_accuracy': 0.9359330949948927, 'total': 14895}


In [7]:
# OVERVIEW: Optionally run three class-level unlearning update-scope baselines.
def forget_sets(records, label):
    forget=[r for r in records if r.source == 'AOL' and r.original_label == label]
    return forget,[r for r in records if r not in forget]
def choose_label():
    valid=[label for label in sorted(KNOWN_LABELS) if all(forget_sets(split,label)[0] for split in (train_records,val_records,test_records))]
    if not valid: raise ValueError('No AOL label occurs in all splits.')
    return random.Random(SEED).choice(valid)
def set_scope(branch, scope):
    for p in branch.parameters(): p.requires_grad=False
    modules={'head_only':[branch.classifier],'last_encoder_block':[branch.encoder.conv4,branch.encoder.conv4_4,branch.encoder.batch_norm4,branch.encoder.fc,branch.classifier],'full_encoder_and_head':[branch.encoder,branch.classifier]}[scope]
    for module in modules:
        for p in module.parameters(): p.requires_grad=True
def set_unlearning_train_mode(branch, scope):
    branch.train()
    if scope == 'head_only':
        branch.encoder.eval()  # Không cập nhật BatchNorm/Dropout encoder khi chỉ train MLP.
    elif scope == 'last_encoder_block':
        branch.encoder.eval()
        for module in (branch.encoder.conv4, branch.encoder.conv4_4, branch.encoder.batch_norm4, branch.encoder.max_pool_4, branch.encoder.dropout4, branch.encoder.fc):
            module.train()
        branch.classifier.train()
def forget_rate(branch, loader):
    branch.eval(); total=unknown=0
    with torch.no_grad():
        for batch in loader:
            features,mask,_=move(batch); logits,_=branch(features,mask); unknown += (torch.softmax(logits,1)[:,1] >= BEST_UNKNOWN_THRESHOLD).sum().item(); total += features.size(0)
    return unknown/max(total,1)
if RUN_UNLEARNING:
    _valid_labels=[label for label in sorted(KNOWN_LABELS) if all(forget_sets(split,label)[0] for split in (train_records,val_records,test_records))]
    print(f'Candidate forget labels: {len(_valid_labels)}/{len(KNOWN_LABELS)} AOL labels')
    forget_label=FORGET_LABEL or choose_label(); results=[]
    print(f'FORGET_LABEL={forget_label!r} (manual={FORGET_LABEL is not None})')
    ftr,rtr=forget_sets(train_records,forget_label); fva,rva=forget_sets(val_records,forget_label); fte,rte=forget_sets(test_records,forget_label)
    if not all((ftr,rtr,fva,rva,fte,rte)): raise ValueError(f'Empty Df/Dr for {forget_label}')
    print(f'Df/Dr sizes: train {len(ftr)}/{len(rtr)}, val {len(fva)}/{len(rva)}, test {len(fte)}/{len(rte)}')
    print(f'BEST_UNKNOWN_THRESHOLD={BEST_UNKNOWN_THRESHOLD:.2f}')
    _pre_forget={name: forget_rate(model,make_loader(recs,False)) for name,recs in [('Df-train',ftr),('Df-val',fva),('Df-test',fte)]}
    _pre_retain={name: evaluate(model,make_loader(recs,False),BEST_UNKNOWN_THRESHOLD) for name,recs in [('Dr-train',rtr),('Dr-val',rva),('Dr-test',rte)]}
    print('Pre-unlearning forget_rate (Df -> unknown):', {k: round(v,4) for k,v in _pre_forget.items()})
    print('Pre-unlearning retain:', {k: {'acc': round(v['accuracy'],4), 'bal_acc': round(v['balanced_accuracy'],4), 'known_rec': round(v['known_recall'],4), 'unk_rec': round(v['unknown_recall'],4)} for k,v in _pre_retain.items()})
    for scope in UNLEARNING_SCOPES:
        branch=copy.deepcopy(model).to(DEVICE); set_scope(branch,scope); opt=torch.optim.AdamW([p for p in branch.parameters() if p.requires_grad],lr=UNLEARNING_LR,weight_decay=WEIGHT_DECAY); ce=nn.CrossEntropyLoss()
        f_loader,r_loader=make_loader(ftr,True),make_loader(rtr,True)
        best=copy.deepcopy(branch.state_dict()); best_score=-1.
        for _epoch in range(UNLEARNING_EPOCHS):
            set_unlearning_train_mode(branch, scope); fi=iter(f_loader); ri=iter(r_loader)
            for _ in range(max(len(f_loader),len(r_loader))):
                try: fb=next(fi)
                except StopIteration: fi=iter(f_loader); fb=next(fi)
                try: rb=next(ri)
                except StopIteration: ri=iter(r_loader); rb=next(ri)
                fx,fm,_=move(fb); rx,rm,ry=move(rb); opt.zero_grad(set_to_none=True); fl,_=branch(fx,fm); rl,_=branch(rx,rm); loss=ce(fl,torch.ones(fl.size(0),dtype=torch.long,device=DEVICE))+RETAIN_LOSS_WEIGHT*ce(rl,ry); loss.backward(); nn.utils.clip_grad_norm_([p for p in branch.parameters() if p.requires_grad], max_norm=5.0); opt.step()
            _fr=forget_rate(branch,make_loader(fva,False)); _bal=evaluate(branch,make_loader(rva,False),BEST_UNKNOWN_THRESHOLD)['balanced_accuracy']; score=.5*(_fr+_bal)
            if score>best_score: best_score=score; best=copy.deepcopy(branch.state_dict())
            print(f'  [{scope}] epoch {_epoch+1}/{UNLEARNING_EPOCHS}: loss_last={loss.item():.4f} | Df-val forget_rate={_fr:.4f} | Dr-val bal_acc={_bal:.4f} | score={score:.4f} (best={best_score:.4f})')
        branch.load_state_dict(best); result={'scope':scope,'forget_label':forget_label,'forget_as_unknown_test':forget_rate(branch,make_loader(fte,False)),'retain_test':evaluate(branch,make_loader(rte,False),BEST_UNKNOWN_THRESHOLD)}
        print(f'  [{scope}] DONE: Df-test forget {_pre_forget["Df-test"]:.4f} -> {result["forget_as_unknown_test"]:.4f} | Dr-test bal_acc {_pre_retain["Dr-test"]["balanced_accuracy"]:.4f} -> {result["retain_test"]["balanced_accuracy"]:.4f} | Dr-test acc {_pre_retain["Dr-test"]["accuracy"]:.4f} -> {result["retain_test"]["accuracy"]:.4f}')
        directory=UNLEARNING_ROOT/scope; directory.mkdir(parents=True,exist_ok=True); torch.save({'model_state_dict':branch.state_dict(),**result},directory/'unlearned_model.pt'); write_json(directory/'result.json',result); (directory/'README.md').write_text(f'# {scope} unlearning baseline\n\nForget AOL label: `{forget_label}`. Df is trained toward binary unknown (1); Dr retains its original binary labels.\n', encoding='utf-8'); results.append(result)
    write_json(UNLEARNING_ROOT/'summary.json',{'forget_label':forget_label,'best_unknown_threshold':BEST_UNKNOWN_THRESHOLD,'split_sizes':{'ftr':len(ftr),'rtr':len(rtr),'fva':len(fva),'rva':len(rva),'fte':len(fte),'rte':len(rte)},'pre_unlearning':{'forget_rate':_pre_forget,'retain':{k: {m: v[m] for m in ('loss','accuracy','known_recall','unknown_recall','balanced_accuracy','total')} for k,v in _pre_retain.items()}},'results':results})
    print(results)
else:
    print('RUN_UNLEARNING=False; base model only.')



Candidate forget labels: 50/50 AOL labels
FORGET_LABEL='starter_kit_for_aol_high_speed' (manual=False)
Df/Dr sizes: train 85/12574, val 15/2218, test 100/14795
BEST_UNKNOWN_THRESHOLD=0.50
Pre-unlearning forget_rate (Df -> unknown): {'Df-train': 0.0, 'Df-val': 0.0, 'Df-test': 0.06}
Pre-unlearning retain: {'Dr-train': {'acc': 0.9883, 'bal_acc': 0.988, 'known_rec': 0.9872, 'unk_rec': 0.9888}, 'Dr-val': {'acc': 0.9833, 'bal_acc': 0.9811, 'known_rec': 0.9749, 'unk_rec': 0.9873}, 'Dr-test': {'acc': 0.9521, 'bal_acc': 0.9354, 'known_rec': 0.8878, 'unk_rec': 0.983}}
  [head_only] epoch 1/3: loss_last=0.1292 | Df-val forget_rate=1.0000 | Dr-val bal_acc=0.9819 | score=0.9910 (best=0.9910)
  [head_only] epoch 2/3: loss_last=0.0533 | Df-val forget_rate=1.0000 | Dr-val bal_acc=0.9841 | score=0.9921 (best=0.9921)
  [head_only] epoch 3/3: loss_last=0.0040 | Df-val forget_rate=1.0000 | Dr-val bal_acc=0.9783 | score=0.9891 (best=0.9921)
  [head_only] DONE: Df-test forget 0.0600 -> 1.0000 | Dr-test bal_